# Training YOLO di Colab / Kaggle

Notebook ini training model YOLO dari ZIP export Label Studio (format YOLO), memakai fungsi prepare dataset milik repo ini supaya hasilnya identik dengan training lewat Streamlit.

Alur:

1. Streamlit: capture image dan sync ke Label Studio
2. Label Studio: labeling, lalu export YOLO (ZIP) dari tab Label Studio
3. Notebook ini: training di GPU gratis
4. Download `best.pt`, pilih di sidebar Streamlit lewat opsi `Path custom`

Colab: pastikan `Runtime > Change runtime type > T4 GPU`.
Kaggle: `Settings > Accelerator > GPU T4 x2`, dan `Internet` harus ON.

## 1. Install dan cek GPU

In [ ]:
!pip install -q ultralytics

import torch

HAS_GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "TIDAK ADA - training bakal sangat lambat")


## 2. Clone repo

Dipakai untuk `prepare_label_studio_yolo_dataset`, yang mengubah export Label Studio jadi struktur `images/train`, `images/val`, dan `data.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("/content")
REPO_DIR = WORK_DIR / "yolo"
REPO_URL = os.environ.get("YOLO_REPO_URL", "https://github.com/faprikaa/yolo.git")

if not REPO_DIR.exists():
    !git clone -q {REPO_URL} {REPO_DIR}

sys.path.insert(0, str(REPO_DIR / "src"))
print("repo:", REPO_DIR)


## 3. Ambil ZIP export Label Studio

Upload sekali, dipakai terus:

- **Colab**: ZIP disimpan di Google Drive `MyDrive/yolo_datasets`. Kalau folder itu masih kosong, cell ini membuka dialog upload lalu menyalin ZIP-nya ke Drive, jadi run berikutnya langsung jalan tanpa upload. ZIP paling baru yang dipakai.
- **Kaggle**: upload ZIP sekali lewat `Add Data > Upload Dataset`, lalu ganti `KAGGLE_ZIP_PATH` ke path di `/kaggle/input/...`. Notebook berikutnya tinggal attach dataset yang sama.

Habis labeling batch baru: export ulang dari Label Studio, taruh ZIP barunya di folder yang sama.

In [ ]:
import shutil

KAGGLE_ZIP_PATH = "/kaggle/input/nut-dataset/export.zip"
DRIVE_DATASET_DIR = "/content/drive/MyDrive/yolo_datasets"

if IS_KAGGLE:
    ZIP_PATH = Path(KAGGLE_ZIP_PATH)
else:
    from google.colab import drive, files

    drive.mount("/content/drive")
    drive_dir = Path(DRIVE_DATASET_DIR)
    drive_dir.mkdir(parents=True, exist_ok=True)

    # ponytail: pakai ZIP termuda di Drive, jadi export baru cukup di-drop ke folder itu
    existing = sorted(drive_dir.glob("*.zip"), key=lambda path: path.stat().st_mtime)
    if existing:
        ZIP_PATH = existing[-1]
    else:
        uploaded = files.upload()
        ZIP_PATH = drive_dir / next(iter(uploaded))
        shutil.copy2(Path(next(iter(uploaded))), ZIP_PATH)
        print(f"Disimpan ke Drive: {ZIP_PATH}")

assert ZIP_PATH.exists(), f"ZIP tidak ditemukan: {ZIP_PATH}"
print("dataset zip:", ZIP_PATH)

## 4. Siapkan dataset

Cek output `split_counts` dan `labeled_images`. Kalau `labeled_images` jauh lebih kecil dari total image, berarti ada image yang belum dilabeli di Label Studio.

In [ ]:
from yolo_dashboard.training import prepare_label_studio_yolo_dataset

prepared = prepare_label_studio_yolo_dataset(
    archive_path=ZIP_PATH,
    dataset_root=WORK_DIR / "datasets",
    train_split=0.8,
)

print("data.yaml :", prepared.data_yaml_path)
print("classes   :", prepared.class_names)
print("split     :", prepared.split_counts)
print("berlabel  :", prepared.labeled_images)


## 5. Training

`imgsz=640` jangan diturunkan untuk objek kecil seperti nut. Kalau kehabisan VRAM, turunkan `BATCH` dulu, bukan `IMGSZ`.

In [ ]:
from ultralytics import YOLO

BASE_MODEL = "yolo11n.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
RUN_NAME = "nut_inspection"

model = YOLO(BASE_MODEL)
results = model.train(
    data=str(prepared.data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=25,
    device=0 if HAS_GPU else "cpu",
    project=str(WORK_DIR / "runs"),
    name=RUN_NAME,
    exist_ok=True,
)

BEST_MODEL = Path(results.save_dir) / "weights" / "best.pt"
print("best model:", BEST_MODEL)


## 6. Cek hasil

Patokan kasar: `mAP50` di atas 0.85 sudah layak dipakai. Kalau salah satu class jauh lebih jelek, tambah data untuk class itu, bukan tambah epoch.

In [ ]:
metrics = YOLO(BEST_MODEL).val(data=str(prepared.data_yaml_path), imgsz=IMGSZ)

print(f"mAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")
for index, class_name in enumerate(prepared.class_names):
    print(f"  {class_name:<16} mAP50={metrics.box.ap50[index]:.4f}")


## 7. Download `best.pt`

Di Streamlit lokal: sidebar > sumber model `Path custom` > isi path file `best.pt` yang sudah didownload.

In [ ]:
import shutil

target = WORK_DIR / f"{RUN_NAME}_best.pt"
shutil.copy2(BEST_MODEL, target)

if IS_KAGGLE:
    print(f"Selesai. Ambil dari tab Output: {target}")
else:
    from google.colab import files

    files.download(str(target))
